## First test of VA in chamber

In [1]:
# Imports
import numpy as np
import time

from mcg_lab_vna_control import connect_vna
from mcg_lab_triaxial_positioner_control import TriaxialPositioner

# Transparent conection: the library knows witch driver to use
vna = connect_vna(brand='keysight')


In [2]:
########## Parameters ##########
# Triaxial Positioner Port
PORT = "COM3"

# VNA Parameters
ifbw = 1e3
pow_dbm = 0
f_start_list = [10e9,12e9,14e9] # Segmented f_start lis (in Hz)
f_stop_list = [11e9,13e9,15e9] # Segmented f_stop lis (in Hz)
points_list = [1001] # Segmented points list

# Folder name for saving measurement files
folder_name = 'VA_test'

### Basic VNA Config to see Time-Response of the Channel

In [3]:
vna.preset()

vna.configure_sparam(sparam_str = 'S21', trace = 1, f_start = 10e9, f_stop = 15e9, points = 5001, ifbw = 10e3, pow_dbm = pow_dbm)
vna.configure_sparam(sparam_str = 'S22', trace = 2, f_start = 10e9, f_stop = 15e9, points = 5001, ifbw = 10e3, pow_dbm = pow_dbm)

vna.set_timedomain(trace = 1)
vna.set_timedomain(trace = 2, window='hamming')

vna.run_cont()

time.sleep(2)

vna.auto_scale_all()

Measurement time 0.502209354199 seconds
Measurement time 0.502209354199 seconds


In [4]:
vna.set_Xscale(xmin = 150, xmax = 190, mode = 'dist', trace = 1)
vna.set_Xscale(xmin = 0, xmax = 20, mode = 'dist', trace = 2)
vna.auto_scale_all()

### Calibrate the VNA for Segmented Measurements

In [3]:
vna.preset()

vna.configure_segmented_sparam(sparam_str = 'S21', trace = 1, 
                               f_start_list = f_start_list, f_stop_list = f_stop_list, 
                               points_list = points_list, ifbw = ifbw, pow_dbm = pow_dbm)

vna.configure_segmented_sparam(sparam_str = 'S22', trace = 2, 
                               f_start_list = f_start_list, f_stop_list = f_stop_list,
                               points_list = points_list, ifbw = ifbw, pow_dbm = pow_dbm)

sweep_time = vna.query_sweep_time()
time.sleep(2*sweep_time + 1)  # Wait for the sweep to complete
vna.auto_scale_all()
vna.run_cont()

Measurement time 2.89627171382 seconds
Measurement time 2.89627171382 seconds
Measurement time 2.89627171382 seconds


#### Port 1:

In [6]:
vna.init_SOL_calibration(port = 1, conn_type = '2.92', conn_gender = 'f', calkit = 'ZV-Z129')

Steps needed for SOL calibration Port 1: 3
Step 1: "Connect 2.92 MM FEMALE [OPEN] to port 1"
Step 2: "Connect 2.92 MM FEMALE [SHORT] to port 1"
Step 3: "Connect 2.92 MM FEMALE LOAD [MATCH] to port 1"


In [9]:
# Do it with all calibrators numbers
vna.measure_SOL_calibrator(3)

Measurement of SOL calibrator step 3 completed


In [10]:
# Apply calibration
vna.apply_calibration()

#### Port 2:

In [11]:
vna.init_SOL_calibration(port = 2, conn_type = '2.92', conn_gender = 'f', calkit = 'ZV-Z129')

Steps needed for SOL calibration Port 2: 3
Step 1: "Connect 2.92 MM FEMALE [OPEN] to port 2"
Step 2: "Connect 2.92 MM FEMALE [SHORT] to port 2"
Step 3: "Connect 2.92 MM FEMALE LOAD [MATCH] to port 2"


In [14]:
# Do it with all calibrators numbers
vna.measure_SOL_calibrator(3)

Measurement of SOL calibrator step 3 completed


In [15]:
# Apply calibration
vna.apply_calibration()

### Measure S21 of Through for back-to-back calibration

In [ ]:
# Measureme S21 of Through
sparam = vna.measure_trace_sparam(trace=1)
vna.auto_scale_all()

vna.save_s1p(filename= 'b2b_cal.s1p', folder = folder_name, freqs = vna.get_freq_vector(), sdata = sparam, sparam_str = 'S21')

✅ File 'b2b_cal.s1p' (S21) generated correctly.


### VA Grid Measurement Loop

In [3]:
########## Parameters ##########
setup_number = 1 # setup number (to have different setup trials)
va_vert_pos = 0 # 0 | 1, idx position in vertical of antenna of the VA  
va_deg = 0 # degrees of boresight of VA antenna
ue_deg = 0

va_Nh = 30 # VA horizontal (Nh) positions
va_Nv = 15 # VA vertical (Nv) positions
va_deltah = 10.0 # mm of Delta H
va_deltav = 10.0 # mm of Delta V

In [7]:
with TriaxialPositioner(port=PORT) as pos:
    print("Connected to C4 Controller.")
    
    # Configure velocity (optional)
    pos.set_velocity(min_speed=200, max_speed=1500, slope=2000)

    pos.home_all_axis()

    # pos.move_axes_absolute(z=90, unit='mm')

    # pos.move_axes_absolute(z=100, unit='mm')



    

Connected to C4 Controller.

[WARNING] Hardware limit switch hit during motion. Initiating automated recovery...
   [RECOVERY] Axis X limit triggered. Auto-bouncing back by 5.0 mm...
   [RECOVERY] Hardware limit protection re-enabled.
[INFO] Automatic limit recovery successful. The system is ready.

[WARNING] Hardware limit switch hit during motion. Initiating automated recovery...
   [RECOVERY] Axis Y limit triggered. Auto-bouncing back by 5.0 mm...
   [RECOVERY] Hardware limit protection re-enabled.
[INFO] Automatic limit recovery successful. The system is ready.

[WARNING] Hardware limit switch hit during motion. Initiating automated recovery...
   [RECOVERY] Axis Z limit triggered. Auto-bouncing back by 5.0 mm...
   [RECOVERY] Hardware limit protection re-enabled.
[INFO] Automatic limit recovery successful. The system is ready.


In [5]:
# The 'with' statement automatically handles connection and safe disconnection
with TriaxialPositioner(port=PORT) as pos:
    print("Connected to C4 Controller.")
    
    # Configure velocity (optional)
    pos.set_velocity(min_speed=200, max_speed=1500, slope=2000)

    # Move to the home position (0,0,0) before starting the grid scan
    pos.home_all_axis()

    # Establish Absolute Zero
    print("Defining current location as Origin (0,0,0)...")
    pos.define_global_coord(x=0.0, y=0.0, z=0.0, unit='mm')

    # Executing a 2D Grid Scan using Generators
    print("Starting 2D Grid Scan...")
    grid_iterator = pos.generate_grid_2d(
        axis1='Y', 
        axis2='Z', 
        points1=va_Nh, 
        points2=va_Nv, 
        spacing1=va_deltah, 
        spacing2=va_deltav, 
        unit='mm'
    )
    
    for step in grid_iterator:
        print(f"Arrived at Point {step['point']}/{step['total']} | Y: {step['Y']}mm, Z: {step['Z']}mm")

        step_i = step['i']
        step_j = step['j'] + va_vert_pos * va_Nv

        # Current Step Filenames
        filename_s21 = f"setup_{setup_number}_S21_nv_{step_j}_nh_{step_i}_VA_{va_deg}_deg_UE_{ue_deg}deg"
        filename_s22 = f"setup_{setup_number}_S22_nv_{step_j}_nh_{step_i}_VA_{va_deg}_deg_UE_{ue_deg}deg"

        # Measure S21 (Bi BS-UE)
        sparam = vna.measure_trace_sparam(trace=1)
        vna.auto_scale_all()

        vna.save_s1p(filename = filename_s21, folder = folder_name, freqs = vna.get_freq_vector(), sdata = sparam, sparam_str = 'S21')

        time.sleep(1)

        # Measure S22 (Mono UE-UE)
        sparam = vna.measure_trace_sparam(trace=2)
        vna.auto_scale_all()

        vna.save_s1p(filename = filename_s22, folder = folder_name, freqs = vna.get_freq_vector(), sdata = sparam, sparam_str = 'S22')

        time.sleep(1) 
    
    print("Grid scan complete!")

    # Return Home
    print("Returning to origin...")
    pos.move_axes_absolute(x=0.0, y=0.0, z=0.0, wait_to_finish=True, unit='mm')

    print("Measuring the first VA position (0,0) to evaluate the Drift during the measurements loop")

    filename_s21 = f"extra_setup_{setup_number}_S21_nv_0_nh_0_VA_{va_deg}_deg_UE_{ue_deg}deg"
    filename_s22 = f"extra_setup_{setup_number}_S22_nv_0_nh_0_VA_{va_deg}_deg_UE_{ue_deg}deg"

    # Measure S21 (Bi BS-UE)
    sparam = vna.measure_trace_sparam(trace=1)
    vna.auto_scale_all()

    vna.save_s1p(filename = filename_s21, folder = folder_name, freqs = vna.get_freq_vector(), sdata = sparam, sparam_str = 'S21')

    time.sleep(1)

    # Measure S22 (Mono UE-UE)
    sparam = vna.measure_trace_sparam(trace=2)
    vna.auto_scale_all()

    vna.save_s1p(filename = filename_s22, folder = folder_name, freqs = vna.get_freq_vector(), sdata = sparam, sparam_str = 'S22')

    time.sleep(1) 

print("Routine finished and hardware disconnected safely.")





Connected to C4 Controller.

[WARNING] Hardware limit switch hit during motion. Initiating automated recovery...
   [RECOVERY] Axis X limit triggered. Auto-bouncing back by 5.0 mm...
   [RECOVERY] Hardware limit protection re-enabled.
[INFO] Automatic limit recovery successful. The system is ready.

[WARNING] Hardware limit switch hit during motion. Initiating automated recovery...
   [RECOVERY] Axis Y limit triggered. Auto-bouncing back by 5.0 mm...
   [RECOVERY] Hardware limit protection re-enabled.
[INFO] Automatic limit recovery successful. The system is ready.

[WARNING] Hardware limit switch hit during motion. Initiating automated recovery...
   [RECOVERY] Axis Z limit triggered. Auto-bouncing back by 5.0 mm...
   [RECOVERY] Hardware limit protection re-enabled.
[INFO] Automatic limit recovery successful. The system is ready.
Defining current location as Origin (0,0,0)...
Starting 2D Grid Scan...
Arrived at Point 1/450 | Y: 0.0mm, Z: 0.0mm
✅ File 'setup_1_S21_nv_0_nh_0_VA_0_deg_U

KeyboardInterrupt: 